In [1]:
import os
import sys
import zipfile
from pathlib import Path

# Extract Team A code
with zipfile.ZipFile("/content/jiayi_team_a_selfplay.zip") as z:
    z.extractall("/content")

# Extract official SDK files from our prepared ZIP
SDK_DIR = Path("/content/ptcg_sdk")
SDK_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("/content/ptcg_sdk.zip") as z:
    z.extractall(SDK_DIR)

# Configure paths
REPO_DIR = Path("/content")

sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(SDK_DIR))

os.environ["PTCG_SDK_DIR"] = str(SDK_DIR)
os.environ["PTCG_CG_DIR"] = str(SDK_DIR / "cg")

os.chdir(REPO_DIR)

# Verify required files
assert (REPO_DIR / "team_a/src/models/jiayi/aj1_bc.json").exists()
assert (REPO_DIR / "team_a/src/models/jiayi/aj2_bc.json").exists()
assert (SDK_DIR / "cg/libcg.so").exists()
assert (SDK_DIR / "deck.csv").exists()

import cg.game

print("Self-play environment ready!")

Self-play environment ready!


In [2]:
import os
import sys
import platform
import zipfile
from pathlib import Path
from pathlib import Path
from collections import Counter
import importlib
from statistics import mean
import pandas as pd
from IPython.display import display

In [3]:
repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "team_a/configs/evaluation.json").is_file())
sys.path.insert(0, str(repo))
from team_a.src.common_runtime import agent_import_path
from team_a.src.models.jiayi.model_1 import Model as Linear
from team_a.src.models.jiayi.model_2 import Model as MLP
from team_a.src.models.jiayi.self_play import SHAPING_SCALE, play_battle

SDK_DIR = os.environ.get("PTCG_SDK_DIR")
if not SDK_DIR:
    print("Blocked: set PTCG_SDK_DIR to the official sample_submission directory containing cg/ and deck.csv.")
    print("No native battles executed.")

In [4]:
if SDK_DIR:
    sdk_path = Path(SDK_DIR).expanduser().resolve()
    deck_path = sdk_path / "deck.csv"
    deck = [int(line) for line in deck_path.read_text().splitlines() if line.strip()]
    if len(deck) != 60:
        raise ValueError("Expected the official 60-card sample deck")
    model_dir = repo / "team_a/src/models/jiayi"
    specs = [("A-J1", Linear, model_dir / "aj1_bc.json"),
             ("A-J2", MLP, model_dir / "aj2_bc.json")]
    epsilon = 0.1
    seeds = (61000, 61001, 61002, 61003)
    print("Python", sys.version.split()[0], "platform", sys.platform)
    print("Seeds", seeds, "epsilon", epsilon, "shaping scale", SHAPING_SCALE)
    battles, decisions = [], []
    with agent_import_path(sdk_path):
        for name in ("cg.game", "cg.sim"):
            module = importlib.import_module(name)
            expected = sdk_path / (name.replace(".", "/") + ".py")
            if Path(module.__file__).resolve() != expected:
                raise RuntimeError("Restart the kernel to load cg from the configured SDK directory")
        for name, loader, path in specs:
            model = loader(path)
            for index, seed in enumerate(seeds):
                result, trace = play_battle((model, model), deck, seed, epsilon, bool(index % 2))
                counts = Counter(d["source"] for d in trace)
                battles.append(dict(result, policy=name, learned=counts["learned"], fallback=counts["heuristic"]))
                decisions.extend(dict(d, policy=name, seed=seed) for d in trace)
    display(pd.DataFrame(battles).reindex(columns=["policy", "python_seed", "reverse_player", "status", "result_code",
                                                    "steps", "learned", "fallback", "terminal_returns", "shaped_returns",
                                                    "invalid_responses", "engine_rejections", "error_message"]))

Python 3.13.18 platform linux
Seeds (61000, 61001, 61002, 61003) epsilon 0.1 shaping scale 0.25


,policy,python_seed,reverse_player,status,result_code,steps,learned,fallback,terminal_returns,shaped_returns,invalid_responses,engine_rejections,error_message
0,A-J1,61000,False,passed,1,79,42,37,"[-1, 1]","[-1.0, 1.0]",0,0,
1,A-J1,61001,True,passed,0,73,46,27,"[1, -1]","[1.0, -1.0]",0,0,
2,A-J1,61002,False,passed,1,22,12,10,"[-1, 1]","[-1.0, 1.0]",0,0,
3,A-J1,61003,True,passed,0,30,19,11,"[1, -1]","[1.0, -1.0]",0,0,
4,A-J2,61000,False,passed,0,16,8,8,"[1, -1]","[1.0, -1.0]",0,0,
5,A-J2,61001,True,passed,1,19,11,8,"[-1, 1]","[-1.0, 1.0]",0,0,
6,A-J2,61002,False,passed,0,45,25,20,"[1, -1]","[1.0, -1.0]",0,0,
7,A-J2,61003,True,passed,1,22,12,10,"[-1, 1]","[-1.0, 1.0]",0,0,


In [5]:
if SDK_DIR:
    comparison = []
    for name, _, _ in specs:
        completed = [d for d in decisions if d["policy"] == name and "terminal_return" in d]
        if not completed:
            continue
        for scheme in ("terminal", "shaped"):
            comparison.append({"policy": name, "scheme": scheme, "accepted_decisions": len(completed),
                               "nonzero_rewards": sum(d[f"{scheme}_reward"] != 0 for d in completed),
                               "mean_absolute_decision_reward": mean(abs(d[f"{scheme}_reward"]) for d in completed),
                               "mean_absolute_return_shift": mean(abs(d[f"{scheme}_return"] - d["terminal_return"]) for d in completed)})
    if comparison:
        display(pd.DataFrame(comparison))
        examples = [next((d for d in decisions if d["source"] == source and "terminal_return" in d), None)
                    for source in ("learned", "heuristic")]
        display(pd.DataFrame([d for d in examples if d is not None]))
    else:
        print("No valid completed battles to compare.")

,policy,scheme,accepted_decisions,nonzero_rewards,mean_absolute_decision_reward,mean_absolute_return_shift
0,A-J1,terminal,204,4,0.019608,0.000000
1,A-J1,shaped,204,14,0.024918,0.019199
2,A-J2,terminal,102,4,0.039216,0.000000
3,A-J2,shaped,102,9,0.049837,0.010621


,source,probability,log_probability,learned_probability,step,seat,action,legal_option_count,prize_lead_seat0,terminal_reward,shaped_reward,terminal_return,shaped_return,policy,seed
0,learned,0.196667,-1.626245,0.2,5,0,[2],6,0.0,0,0.0,-1,-1.0,A-J1,61000
1,heuristic,NaN,NaN,NaN,0,0,[0],2,0.0,0,0.0,-1,-1.0,A-J1,61000
